In [2]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [1]:
import re

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)   # Remove HTML tags
    text = re.sub(r"\r\n|\r|\n", " ", text)  # Remove line breaks
    text = re.sub(r"\s+", " ", text)     # Normalize whitespace
    text = text.strip()

    return text

In [3]:
model = T5ForConditionalGeneration.from_pretrained("../models/saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("../models/saved_summary_model/")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [4]:
def summarize_dialogue(dialogue):
    dialogue = "summarize: " + clean_data(dialogue)

    # tokenization
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )
    
    # generate summary :- token id's will be generated
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # token id's to summary
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)

    return summary

### Testing on Dialogue

In [5]:
test_dialogue = """
Sarah: Hey John, did you finish the project report?
John: Almost. I completed the analysis section yesterday.
Sarah: Great. What about the introduction and conclusion?
John: I finished the introduction this morning, but I still need to write the conclusion.
Sarah: When do you think you'll finish it?
John: I should be able to complete it by tonight and send the final report tomorrow morning.
Sarah: Perfect. I'll review it once you send it.
"""

In [7]:
summary = summarize_dialogue(test_dialogue)

print("Summary is :- ", summary)

Summary is :-  John finished the project report yesterday. Sarah will send the final report tomorrow morning.


### Test on Multiple Unseen Dialogues

In [8]:
test_dialogue_1 = """
Manager: We have a team meeting scheduled for Friday at 10 AM.
Alex: I won't be available at that time because I have a doctor's appointment.
Manager: That's fine. Can you join the meeting at 2 PM instead?
Alex: Yes, 2 PM works for me.
Manager: Great. I'll move the meeting to Friday at 2 PM.
"""

test_dialogue_2 = """
Emma: We need to buy some groceries for dinner.
Tom: What should we get?
Emma: We need vegetables, rice, and some chicken.
Tom: I'll buy the vegetables and rice.
Emma: Great. I'll pick up the chicken on my way home.
Tom: Perfect. Let's meet at home around 7 PM.
"""

test_dialogue_3 = """
Rahul: Have you completed the machine learning project?
Amit: I finished the model training yesterday.
Rahul: What about the documentation?
Amit: I'm working on it now and should finish it by tomorrow.
Rahul: Okay. I'll prepare the presentation after you finish the documentation.
Amit: Sounds good. I'll send it to you tomorrow evening.
"""

In [9]:
summary_1 = summarize_dialogue(test_dialogue_1)
summary_2 = summarize_dialogue(test_dialogue_2)
summary_3 = summarize_dialogue(test_dialogue_3)

print("SUMMARY 1:")
print(summary_1)

print("\nSUMMARY 2:")
print(summary_2)

print("\nSUMMARY 3:")
print(summary_3)

SUMMARY 1:
Manager has a team meeting scheduled for Friday at 10 AM. Alex won't be available at that time because she has a doctor's appointment.

SUMMARY 2:
Emma needs to buy groceries for dinner. Tom will pick up the chicken on his way home around 7 PM.

SUMMARY 3:
Amit finished the machine learning project yesterday. He's working on it now and should finish it by tomorrow evening.


### ROUGE Evaluation

In [11]:
from rouge_score import rouge_scorer

In [12]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

In [13]:
reference = "John completed the project and will submit it tomorrow."
generated = "John finished the project and will submit it tomorrow."

scores = scorer.score(reference, generated)

print(scores)

{'rouge1': Score(precision=0.8888888888888888, recall=0.8888888888888888, fmeasure=0.8888888888888888), 'rouge2': Score(precision=0.75, recall=0.75, fmeasure=0.75), 'rougeL': Score(precision=0.8888888888888888, recall=0.8888888888888888, fmeasure=0.8888888888888888)}


### Evaluation on the validation set

In [15]:
from datasets import load_from_disk

train_dataset = load_from_disk("../data/tokenized/train")
val_dataset = load_from_disk("../data/tokenized/val")

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})
Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [18]:
import torch

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

model.eval()

for i in range(50):

    dialogue = val_dataset[i]["dialogue"]
    reference = val_dataset[i]["summary"]

    generated = summarize_dialogue(dialogue)

    scores = scorer.score(reference, generated)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", sum(rouge1_scores) / len(rouge1_scores))
print("ROUGE-2:", sum(rouge2_scores) / len(rouge2_scores))
print("ROUGE-L:", sum(rougeL_scores) / len(rougeL_scores))

ROUGE-1: 0.4447542344474647
ROUGE-2: 0.20259574987865725
ROUGE-L: 0.37936426903354964
